# Figure: Per-Region TXM Histograms

Shows the distribution of recovered transmission values within each anatomical region
(lung / soft tissue / bone), overlaid with the prior value ranges used as optimization
constraints.

Mask channels follow `MASK_GROUPS` order: `['lung', 'soft', 'bone']`.

**Requires:** `nb_run.ipynb`

Output: `build/segmentation_txm_histograms.png`

In [ ]:
%%capture
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    DRIVE_ROOT = "/content/drive/MyDrive/thesis"  # <-- edit if your Drive path differs

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", f"{DRIVE_ROOT}/notebooks/requirements.txt",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "-e", DRIVE_ROOT,
    ])

    # Change to figures/ so load_results() can find the cached data
    os.chdir(f"{DRIVE_ROOT}/figures")


In [ ]:
import sys, os, glob, joblib

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from chest_xray_sim.visualization import plot_segmentation_overlay

def load_results(results_dir="results"):
    hits = sorted(glob.glob(f"{results_dir}/*/imgs.joblib"))
    if not hits:
        raise FileNotFoundError(f"No results in '{results_dir}'. Run nb_run.ipynb first.")
    d = os.path.dirname(hits[-1])
    print(f"Loading from {d}")
    return {
        "txm":        joblib.load(f"{d}/txm.joblib"),
        "masks":      joblib.load(f"{d}/masks.joblib"),
        "priors":     joblib.load(f"{d}/priors.joblib"),
        "seg_labels": joblib.load(f"{d}/seg_labels.joblib"),
    }

In [ ]:
r = load_results()
txm, masks, priors, seg_labels = r["txm"], r["masks"], r["priors"], r["seg_labels"]

print(f"txm shape:    {txm.shape}")
print(f"masks shape:  {masks.shape}")
print(f"seg_labels:   {seg_labels}")
print(f"priors:")
for label, (lo, hi) in zip(seg_labels, priors):
    print(f"  {label}: [{lo:.3f}, {hi:.3f}]")

In [ ]:
# Single sample: TXM image with segmentation overlay + per-region histograms
IDX = 0
colors = ["royalblue", "tomato", "forestgreen"]

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Left: TXM with region overlays
txm_i   = np.asarray(txm[IDX])
masks_i = np.asarray(masks[IDX])

ax[0].imshow(txm_i, cmap="gray", vmin=0, vmax=1)
for k, (label, color) in enumerate(zip(seg_labels, colors)):
    ax[0].imshow(
        np.ma.masked_where(masks_i[k] == 0, np.ones_like(masks_i[k])),
        interpolation="none",
        alpha=0.4 * (masks_i[k] > 0),
        cmap=ListedColormap([color]),
        vmin=0, vmax=1,
    )
    ax[0].plot([], color=color, label=label, linewidth=2)
ax[0].axis("off")
ax[0].legend()
ax[0].set_title(f"Recovered TXM [{IDX}] with segmentation")

# Right: per-region histograms with prior range shading
ax[1].hist(txm_i.flatten(), bins=80, alpha=0.3, color="gray", label="full image")
ax[1].set_box_aspect(1)

for k, (label, color, (lo, hi)) in enumerate(zip(seg_labels, colors, priors)):
    region_vals = txm_i[masks_i[k] > 0]
    if len(region_vals) > 0:
        ax[1].hist(region_vals, bins=40, alpha=0.5, label=label, color=color)
    ax[1].axvspan(lo, hi, alpha=0.08, color=color, label=f"{label} prior")

ax[1].set_xlabel("Transmission value")
ax[1].set_ylabel("Frequency")
ax[1].set_title("Per-region TXM distribution (shading = prior range)")
ax[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig("build/segmentation_txm_histograms.png", bbox_inches="tight")
plt.show()

In [ ]:
# Population view: aggregate per-region values across all images
fig, axes = plt.subplots(1, len(seg_labels), figsize=(5 * len(seg_labels), 5), sharey=True)

for k, (label, color, (lo, hi), ax) in enumerate(zip(seg_labels, colors, priors, axes)):
    all_vals = np.asarray(txm)[np.asarray(masks)[:, k] > 0]
    ax.hist(all_vals, bins=60, color=color, alpha=0.7, density=True)
    ax.axvline(lo, color="black", linestyle="--", label=f"prior min={lo:.2f}")
    ax.axvline(hi, color="black", linestyle=":",  label=f"prior max={hi:.2f}")
    ax.set_title(label)
    ax.set_xlabel("Transmission")
    ax.legend(fontsize=8)
    if k == 0:
        ax.set_ylabel("Density")

fig.suptitle(f"Per-region TXM distributions across {txm.shape[0]} images")
fig.tight_layout()
fig.savefig("build/segmentation_txm_histograms_population.png", bbox_inches="tight")
plt.show()